## Expanded out-of-sample accuracy diagnostics

In [2]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adam

import itertools
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --------------------------------------------------
# Print TensorFlow version
# --------------------------------------------------
print(f"TensorFlow version: {tf.__version__}")

# --------------------------------------------------
# Set random seed for reproducibility
# --------------------------------------------------
SEED = 125

# Python
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# TensorFlow/Keras
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic TensorFlow operations (TF >= 2.9)
tf.config.experimental.enable_op_determinism()

print(f"Random seed set to {SEED}")

TensorFlow version: 2.18.0
Random seed set to 125


## Model-Building Function
Create a function that builds a Keras model based on input parameters. This function allows flexibility in specifying the number of layers, neurons, activation functions, regularization, etc.

In [4]:
def build_model(input_dim,
                nr_neurons,
                reg_param=0.00,
                activation='relu',
                use_batch_norm=True,
                dropout_rate=0.0,
                learning_rate=0.005):
    """
    Builds a customizable neural network model for regression tasks.

    Parameters:
    - input_dim (int): Number of input features.
    - nr_neurons (list): List defining the number of neurons in each hidden layer.
                            Example: [64, 32] for 2 layers with 64 and 32 neurons.
    - reg_param (float): L2 regularization parameter.
    - activation (str): Activation function for hidden layers.
    - use_batch_norm (bool): Whether to use Batch Normalization.
    - dropout_rate (float): Dropout rate for regularization.

    Returns:
    - model (keras.Model): Compiled Keras model.
    """
    model = models.Sequential()

    # Input layer (first hidden layer)
    model.add(layers.Dense(nr_neurons[0],
                           kernel_regularizer=regularizers.l2(reg_param),
                           activation=activation,
                           input_shape=(input_dim,)))

    if use_batch_norm:
        model.add(layers.BatchNormalization())
    if dropout_rate > 0.0:
        model.add(layers.Dropout(dropout_rate))

    # Add remaining hidden layers
    for neurons in nr_neurons[1:]:
        model.add(layers.Dense(neurons,
                               kernel_regularizer=regularizers.l2(reg_param),
                               activation=activation))
        if use_batch_norm:
            model.add(layers.BatchNormalization())
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))

    # Output layer
    model.add(layers.Dense(1))  # Single output for regression

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse',
                  metrics=['mae', 'mse'])

    return model

In [106]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

## Read the data - Gamma

In [109]:
# Load your dataset
df = pd.read_csv("CAT_price_gamma.csv")

print(df.head())

      c         r  kappa  theta  sigma     lambda             D  N         T  \
0  0.05  0.004795    0.2   0.03   0.02  30.210800  1.264946e+10  8  1.127421   
1  0.05  0.054865    0.2   0.03   0.02  39.169999  1.299456e+10  3  1.846108   
2  0.05  0.068230    0.2   0.03   0.02  34.941122  8.628679e+09  8  1.246646   
3  0.05  0.000815    0.2   0.03   0.02  35.785227  1.164647e+10  3  1.729329   
4  0.05  0.055641    0.2   0.03   0.02  32.626761  9.544387e+09  6  0.537173   

         Price  
0  1390.038266  
1   794.831852  
2  1148.438033  
3   936.035903  
4  1266.170898  


In [111]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price']/1000
print(X.head())
print(y.head())

          r     lambda             D  N         T
0  0.004795  30.210800  1.264946e+10  8  1.127421
1  0.054865  39.169999  1.299456e+10  3  1.846108
2  0.068230  34.941122  8.628679e+09  8  1.246646
3  0.000815  35.785227  1.164647e+10  3  1.729329
4  0.055641  32.626761  9.544387e+09  6  0.537173
0    1.390038
1    0.794832
2    1.148438
3    0.936036
4    1.266171
Name: Price, dtype: float64


## Training on Entire Dataset

In [114]:
# Split data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=125
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

import joblib
# Save the scaler
joblib.dump(scaler, 'scaler_gamma.pkl')

print("Min y_train:", np.min(y_train))
print("Max y_train:", np.max(y_train))

Min y_train: 0.0003526900584948981
Max y_train: 1.5997562713872224


In [116]:
# Define different configurations
test_layers = [256, 128, 64, 32]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [118]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_gamma = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_gamma = test_model_gamma.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,             
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
741/750 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.4616 - mae: 1.2633 - mse: 2.4306

2026-07-04 09:39:34.283058: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-04 09:39:34.283545: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2.4572 - mae: 1.2622 - mse: 2.4262 - val_loss: 1.0345 - val_mae: 0.9192 - val_mse: 1.0036
Epoch 2/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.6262 - mae: 1.0629 - mse: 1.5952 - val_loss: 0.9321 - val_mae: 0.9034 - val_mse: 0.9012
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.3574 - mae: 0.9858 - mse: 1.3265 - val_loss: 0.8207 - val_mae: 0.8554 - val_mse: 0.7898
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.1811 - mae: 0.9275 - mse: 1.1502 - val_loss: 0.7263 - val_mae: 0.8081 - val_mse: 0.6954
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.0408 - mae: 0.8729 - mse: 1.0099 - val_loss: 0.6399 - val_mae: 0.7594 - val_mse: 0.6090
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.9343 - mae: 0.8266 - mse: 0.9033 - val_loss: 0.5609 - val_mae: 0.7096 - val_mse: 0.5300
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.8321 - mae: 0.7772 - mse: 0.8012 - va

### Out-of-sample accuracy diagnostics

In [120]:
# Predictions
y_pred = test_model_gamma.predict(X_test, verbose=0).flatten()

# Errors
error_gamma = y_pred - y_test
abs_error_gamma = np.abs(error_gamma)

# Statistics
results_gamma = {
    "Severity": "Gamma",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_gamma),
    "MAE": np.mean(abs_error_gamma),
    "MSE": np.mean(error_gamma**2),
    "RMSE": np.sqrt(np.mean(error_gamma**2)),
    "95% AE": np.quantile(abs_error_gamma, 0.95),
    "99% AE": np.quantile(abs_error_gamma, 0.99),
    "Max AE": np.max(abs_error_gamma),
    "R2": r2_score(y_test, y_pred)
}

results_gamma_df = pd.DataFrame([results_gamma])

print(results_gamma_df)

  Severity  Observations  Training Time (s)      Bias       MAE       MSE  \
0    Gamma        120000        1782.005472  0.000264  0.002737  0.000014   

       RMSE   95% AE    99% AE    Max AE        R2  
0  0.003795  0.00808  0.012855  0.025555  0.999889  


### Operational domain, boundary regions, and extrapolation

In [122]:
r0_boundary = (
    (X_test_raw["r"] <= 0.004) |
    (X_test_raw["r"] >= 0.076)
)

lambda_boundary = (
    (X_test_raw["lambda"] <= 30.5) |
    (X_test_raw["lambda"] >= 39.5)
)

D_boundary = (
    (X_test_raw["D"] <= 7.3e9) |
    (X_test_raw["D"] >= 12.7e9)
)

N_boundary = X_test_raw["N"].isin([0, 12])

T_boundary = (
    (X_test_raw["T"] <= 121.5*1/360) |
    (X_test_raw["T"] >= 688.5*1/360)
)

In [123]:
def subset_stats(mask):

    err = error_gamma[mask]
    ae = abs_error_gamma[mask]

    return {
        "Observations": mask.sum(),
        "MAE": np.mean(ae),
        "RMSE": np.sqrt(np.mean(err**2)),
        "Max AE": np.max(ae)
    }

In [124]:
rows = []

# Full test set
rows.append({
    "Subset":"Full test set",
    **subset_stats(np.ones(len(y_test), dtype=bool))
})

rows.append({
    "Subset":"r0 boundary",
    **subset_stats(r0_boundary)
})

rows.append({
    "Subset":"λ boundary",
    **subset_stats(lambda_boundary)
})

rows.append({
    "Subset":"D boundary",
    **subset_stats(D_boundary)
})

rows.append({
    "Subset":"T boundary",
    **subset_stats(T_boundary)
})

rows.append({
    "Subset":"N ∈ {0,12}",
    **subset_stats(N_boundary)
})

boundary_table = pd.DataFrame(rows)

print(boundary_table)

          Subset  Observations       MAE      RMSE    Max AE
0  Full test set        120000  0.002737  0.003795  0.025555
1    r0 boundary         11877  0.003038  0.004075  0.021078
2     λ boundary         12007  0.002909  0.004044  0.023479
3     D boundary         11864  0.003046  0.004133  0.023907
4     T boundary         12063  0.003276  0.004472  0.023235
5     N ∈ {0,12}         30080  0.002962  0.004073  0.025555


## Read the data - Lognormal

In [56]:
# Load your dataset
df = pd.read_csv("CAT_price_log.csv")

print(df.head())
print(df.shape)

      c         r  kappa  theta  sigma     lambda             D   N         T  \
0  0.05  0.047464    0.2   0.03   0.02  37.595368  1.067337e+10  10  1.971323   
1  0.05  0.005287    0.2   0.03   0.02  30.240079  1.244350e+10  10  1.947998   
2  0.05  0.037334    0.2   0.03   0.02  33.820770  9.731976e+09  10  1.557770   
3  0.05  0.026329    0.2   0.03   0.02  37.278229  1.171317e+10   8  0.815643   
4  0.05  0.046435    0.2   0.03   0.02  31.427952  1.049263e+10  12  0.832221   

         Price  
0   680.036064  
1  1384.209831  
2  1179.769620  
3  1373.370457  
4  1548.602842  
(600000, 10)


In [58]:
# Define your features (X) and labels (y)
# Replace 'feature_columns' and 'target_column' with actual column names
X = df[['r','lambda', 'D', 'N', 'T']]
y = df['Price'] / 1000

print(X.head())
print(y.head())

          r     lambda             D   N         T
0  0.047464  37.595368  1.067337e+10  10  1.971323
1  0.005287  30.240079  1.244350e+10  10  1.947998
2  0.037334  33.820770  9.731976e+09  10  1.557770
3  0.026329  37.278229  1.171317e+10   8  0.815643
4  0.046435  31.427952  1.049263e+10  12  0.832221
0    0.680036
1    1.384210
2    1.179770
3    1.373370
4    1.548603
Name: Price, dtype: float64


## Training on Entire Dataset

In [61]:
# Split data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=125)

# Normalize the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

import joblib
# Save the scaler
joblib.dump(scaler, 'scaler_log.pkl')

print("Min y_train:", np.min(y_train))
print("Max y_train:", np.max(y_train))

Min y_train: 0.0016483922476573647
Max y_train: 1.5996804426516338


In [63]:
# Define different configurations
test_layers = [256, 128, 64, 32]

test_activation = 'relu'
test_l2 = 1e-4
test_dropout = 0.1
test_batch_norm = True
test_learning = 1e-5

In [90]:
import time

EPOCHS = 1000
BATCH_SIZE = 512

# Build the model
test_model_log = build_model(input_dim=X_train.shape[1],
                         nr_neurons=test_layers,
                         reg_param=test_l2,
                         activation=test_activation,
                         use_batch_norm=test_batch_norm,
                         dropout_rate=test_dropout,
                         learning_rate=test_learning)
# Measure training time
start_time = time.perf_counter()

# Train the model on the entire dataset
history_log = test_model_log.fit(X_train, y_train,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=0.2,
                         shuffle=True,         
                         callbacks=[early_stop],
                         verbose=1)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/1000
739/750 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.6110 - mae: 1.3416 - mse: 2.5801

2026-07-02 15:38:00.020347: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-07-02 15:38:00.020593: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2.6046 - mae: 1.3400 - mse: 2.5737 - val_loss: 1.6394 - val_mae: 1.2141 - val_mse: 1.6085
Epoch 2/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.6215 - mae: 1.0791 - mse: 1.5905 - val_loss: 1.2655 - val_mae: 1.0787 - val_mse: 1.2346
Epoch 3/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.3443 - mae: 0.9920 - mse: 1.3134 - val_loss: 1.0794 - val_mae: 1.0010 - val_mse: 1.0485
Epoch 4/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.1743 - mae: 0.9310 - mse: 1.1434 - val_loss: 0.9389 - val_mae: 0.9352 - val_mse: 0.9080
Epoch 5/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.0413 - mae: 0.8775 - mse: 1.0104 - val_loss: 0.8243 - val_mae: 0.8760 - val_mse: 0.7934
Epoch 6/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.9331 - mae: 0.8287 - mse: 0.9022 - val_loss: 0.7193 - val_mae: 0.8168 - val_mse: 0.6885
Epoch 7/1000
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.8370 - mae: 0.7810 - mse: 0.8061 - va

### Out-of-sample accuracy diagnostics

In [92]:
# Predictions
y_pred = test_model_log.predict(X_test, verbose=0).flatten()

# Errors
error_log = y_pred - y_test
abs_error_log = np.abs(error_log)

# Statistics
results_log = {
    "Severity": "Lognormal",
    "Observations": len(y_test),
    "Training Time (s)": training_time,
    "Bias": np.mean(error_log),
    "MAE": np.mean(abs_error_log),
    "MSE": np.mean(error_log**2),
    "RMSE": np.sqrt(np.mean(error_log**2)),
    "95% AE": np.quantile(abs_error_log, 0.95),
    "99% AE": np.quantile(abs_error_log, 0.99),
    "Max AE": np.max(abs_error_log),
    "R2": r2_score(y_test, y_pred)
}

results_log_df = pd.DataFrame([results_log])

print(results_log_df)

    Severity  Observations  Training Time (s)      Bias       MAE       MSE  \
0  Lognormal        120000        2158.520696  0.000094  0.003079  0.000018   

       RMSE    95% AE   99% AE   Max AE        R2  
0  0.004289  0.009283  0.01413  0.03296  0.999843  


### Operational domain, boundary regions, and extrapolation

In [94]:
r0_boundary = (
    (X_test_raw["r"] <= 0.004) |
    (X_test_raw["r"] >= 0.076)
)

lambda_boundary = (
    (X_test_raw["lambda"] <= 30.5) |
    (X_test_raw["lambda"] >= 39.5)
)

D_boundary = (
    (X_test_raw["D"] <= 7.3e9) |
    (X_test_raw["D"] >= 12.7e9)
)

N_boundary = X_test_raw["N"].isin([0, 12])

T_boundary = (
    (X_test_raw["T"] <= 121.5*1/360) |
    (X_test_raw["T"] >= 688.5*1/360)
)

In [95]:
def subset_stats(mask):

    err = error_log[mask]
    ae = abs_error_log[mask]

    return {
        "Observations": mask.sum(),
        "MAE": np.mean(ae),
        "RMSE": np.sqrt(np.mean(err**2)),
        "Max AE": np.max(ae)
    }

In [96]:
rows = []

# Full test set
rows.append({
    "Subset":"Full test set",
    **subset_stats(np.ones(len(y_test), dtype=bool))
})

rows.append({
    "Subset":"r0 boundary",
    **subset_stats(r0_boundary)
})

rows.append({
    "Subset":"λ boundary",
    **subset_stats(lambda_boundary)
})

rows.append({
    "Subset":"D boundary",
    **subset_stats(D_boundary)
})

rows.append({
    "Subset":"T boundary",
    **subset_stats(T_boundary)
})

rows.append({
    "Subset":"N ∈ {0,12}",
    **subset_stats(N_boundary)
})

boundary_table = pd.DataFrame(rows)

print(boundary_table)

          Subset  Observations       MAE      RMSE    Max AE
0  Full test set        120000  0.003079  0.004289  0.032960
1    r0 boundary         12081  0.003450  0.004639  0.032960
2     λ boundary         12265  0.003284  0.004632  0.032960
3     D boundary         11822  0.003276  0.004558  0.032960
4     T boundary         11880  0.003657  0.004986  0.029177
5     N ∈ {0,12}         29941  0.003330  0.004703  0.032960


In [97]:
# Save the final model
test_model_gamma.save("best_NN_gamma.keras")
print("Best model saved as 'best_NN_gamma.keras'")
# Save the final model
test_model_log.save("best_NN_lognormal.keras")
print("Best model saved as 'best_NN_lognormal.keras'")

Best model saved as 'best_NN_gamma.keras'
Best model saved as 'best_NN_lognormal.keras'
